In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
from google.colab import userdata
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

In [ ]:
@tool
def lookup_tour_stop(artist: str) -> str:
    """
    Look up the next city and venue for an artist from a small curated tour calendar.

    Args:
        artist: The artist or band name to search for.
    """
    stops = {
        "breaking benjamin": "Sofia - Arena 8888",
        "placido domingo": "Varna - Palace of Culture and Sports",
        "vassil petrov & jp3": "Shumen - City Stage",
    }
    return stops.get(artist.strip().lower(), "Could not find any tour stops.")

@tool
def estimate_drive_time(origin: str, destination: str) -> str:
    """
    Estimate drive time between cities in Bulgaria.

    Args:
        origin: The departure city.
        destination: The arrival city.
    """
    routes = {
        ("plovdiv", "sofia"): "About 1 hour and 45 minutes.",
        ("shumen", "varna"): "About 1 hour.",
        ("plovdiv", "shumen"): "About 2 hours and 30 minutes."
    }
    key = (origin.strip().lower(), destination.strip().lower())
    return routes.get(key, "Could not estimate the drive time.")

In [ ]:
tools = [lookup_tour_stop, estimate_drive_time]
tools_registry = { t.name: t for t in tools }

openai_default_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key).bind_tools(tools)
prev_messages = [
    SystemMessage("You are a practical live-music concierge. Use tools when they help you give a precise answer."),
    HumanMessage("I'm in Plovdiv on Friday and want to hear live jazz without wasting the whole evening on travel. Check the current mini tour for \"Vassil Petrov & JP3\", figure out the relevant venue, and estimate the drive time."),
]

MAX_ITERATIONS = 100
finished_successfully = False
for i in range(MAX_ITERATIONS):
    reply = openai_default_model.invoke(prev_messages)
    prev_messages.append(reply)

    if not reply.tool_calls:
        finished_successfully = True
        break

    for tool_call in reply.tool_calls:
        tool_call_id = tool_call["id"]
        tool_call_name = tool_call["name"]
        tool_call_args = tool_call["args"]

        result = tools_registry[tool_call_name].invoke(tool_call_args)
        prev_messages.append(ToolMessage(str(result), tool_call_id=tool_call_id))


if not finished_successfully:
    raise RuntimeError(f"Could not finish the interaction within {MAX_ITERATIONS} iterations.")


In [ ]:
print_conversation(prev_messages)